# Banjara AI Training - Pilot Phase

This notebook trains an AI model to translate English to Banjara using your pilot dataset.
We use Meta's **NLLB-200** (No Language Left Behind) model as a base and fine-tune it on your data.

In [ ]:
# 1. Install Dependencies
!pip install transformers datasets evaluate sacrebleu accelerate sentencepiece

In [ ]:
# 2. Unzip Dataset
# Upload your 'dataset.zip' to the Files section on the left before running this!
import os
if not os.path.exists('dataset'):
    !unzip dataset.zip

In [ ]:
# 3. Load Data
from datasets import load_dataset

dataset = load_dataset("csv", data_files={
    "train": "dataset/train/metadata.csv",
    "test": "dataset/test/metadata.csv"
})

print("Sample:", dataset['train'][0])

In [ ]:
# 4. Prepare for Training
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

model_checkpoint = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Source: English (eng_Latn), Target: Rajasthani (raj_Deva) as proxy for Banjara
tokenizer.src_lang = "eng_Latn"
tokenizer.tgt_lang = "raj_Deva"

def preprocess_function(examples):
    inputs = examples["english"]
    targets = examples["banjara"]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(targets, max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

In [ ]:
# 5. Train
training_args = Seq2SeqTrainingArguments(
    output_dir="./banjara_model",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=10,
    predict_with_generate=True,
    fp16=True, # Use GPU acceleration
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

In [ ]:
# 6. Test Translation
from transformers import pipeline

translator = pipeline("translation", model="./banjara_model/checkpoint-500", tokenizer=tokenizer, src_lang="eng_Latn", tgt_lang="raj_Deva")

text = "Where are you going?"
print("Input:", text)
print("Translation:", translator(text)[0]['translation_text'])

In [ ]:
# 7. Save Model to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!cp -r ./banjara_model /content/drive/MyDrive/banjara_model